# AutoPulse — Exploratory Data Analysis (EDA)
**Datasets:** Cleaned `Pakistan_Used_Cars` (`63,735` listings) & `olx_used_cars` (`8,973` listings)  
**Goal:** Analyze price skewness, depreciation dynamics, brand/fuel segmentation, feature correlations, and OLX market signals to guide model selection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'
EVIDENCE_DIR = PROJECT_ROOT / 'reports' / 'evidence'
FIG_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

df = pd.read_csv(DATA_DIR / 'pakistan_cars_clean.csv')
df_olx = pd.read_csv(DATA_DIR / 'olx_clean.csv')

df['price_lacs'] = df['price_pkr'] / 100_000.0
df['log_price'] = np.log1p(df['price_pkr'])
df['car_age'] = 2026 - df['model_year']

print(f'Pakistan Used Cars Clean Shape: {df.shape}')
print(f'OLX Used Cars Clean Shape:      {df_olx.shape}')
df.head(3)

In [ ]:
# FIGURE 1: Raw Price vs. Log-Transformed Price Distribution
raw_skew = df['price_pkr'].skew()
log_skew = df['log_price'].skew()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p99 = df['price_lacs'].quantile(0.99)
sns.histplot(df.loc[df['price_lacs'] <= p99, 'price_lacs'], bins=50, kde=True, color='#1f77b4', ax=axes[0])
axes[0].axvline(df['price_lacs'].median(), color='crimson', linestyle='--', linewidth=1.8,
                label=f"Median: {df['price_lacs'].median():.1f} Lacs")
axes[0].set_title(f'Raw Listing Price Distribution (Skewness = {raw_skew:.2f})')
axes[0].set_xlabel('Price (PKR Lacs — up to 99th Percentile)')
axes[0].set_ylabel('Number of Listings')
axes[0].legend()

sns.histplot(df['log_price'], bins=50, kde=True, color='#2ca02c', ax=axes[1])
axes[1].axvline(df['log_price'].median(), color='crimson', linestyle='--', linewidth=1.8,
                label=f"Median log(Price): {df['log_price'].median():.2f}")
axes[1].set_title(f'Log-Transformed Price Distribution (Skewness = {log_skew:.2f})')
axes[1].set_xlabel('log(Price in PKR)')
axes[1].set_ylabel('Number of Listings')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_price_distribution_skewness.png', bbox_inches='tight')
plt.show()
print(f'Skewness reduced from {raw_skew:.2f} (raw) to {log_skew:.2f} (log-transformed).')

In [ ]:
# FIGURE 2: Price Depreciation by Model Year & Mileage
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

yr_df = df[df['model_year'] >= 2000]
sns.lineplot(
    data=yr_df, x='model_year', y='price_lacs', hue='transmission',
    estimator='median', errorbar=('pi', 50), marker='o', ax=axes[0]
)
axes[0].set_title('Median Price & IQR by Model Year and Transmission')
axes[0].set_xlabel('Model Year')
axes[0].set_ylabel('Median Price (PKR Lacs)')
axes[0].legend(title='Transmission')

bins = [0, 25_000, 50_000, 75_000, 100_000, 150_000, 200_000, 1_000_000]
labels = ['0-25k', '25-50k', '50-75k', '75-100k', '100-150k', '150-200k', '200k+']
df['mileage_bin'] = pd.cut(df['mileage'], bins=bins, labels=labels, include_lowest=True)
sns.boxplot(
    data=df, x='mileage_bin', y='price_lacs', showfliers=False,
    hue='mileage_bin', palette='Blues_r', legend=False, ax=axes[1]
)
axes[1].set_title('Price Distribution Across Mileage Brackets (Outliers Hidden)')
axes[1].set_xlabel('Mileage Driven (km)')
axes[1].set_ylabel('Price (PKR Lacs)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_depreciation_year_mileage.png', bbox_inches='tight')
plt.show()

In [ ]:
# FIGURE 3: Brand Segmentation (Top 12 Makes) & Fuel Type Pricing
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top12_makes = df['make'].value_counts().head(12).index
make_order = (
    df[df['make'].isin(top12_makes)]
    .groupby('make')['price_lacs']
    .median()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=df[df['make'].isin(top12_makes)], y='make', x='price_lacs',
    order=make_order, showfliers=False, hue='make', palette='viridis', legend=False, ax=axes[0]
)
axes[0].set_title('Price Distribution by Top 12 Car Makes (Sorted by Median)')
axes[0].set_xlabel('Price (PKR Lacs)')
axes[0].set_ylabel('Manufacturer (Make)')

fuel_order = df.groupby('fuel_type')['price_lacs'].median().sort_values(ascending=False).index
sns.boxplot(
    data=df, x='fuel_type', y='price_lacs', order=fuel_order,
    showfliers=False, hue='fuel_type', palette='Set2', legend=False, ax=axes[1]
)
axes[1].set_title('Price Distribution by Fuel / Powertrain Type')
axes[1].set_xlabel('Fuel Type')
axes[1].set_ylabel('Price (PKR Lacs)')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_make_and_fuel_pricing.png', bbox_inches='tight')
plt.show()

In [ ]:
# FIGURE 4: Numeric Feature Correlation Heatmap
num_cols = [
    'log_price', 'model_year', 'car_age', 'mileage',
    'engine_cc', 'battery_kwh', 'is_electric_unit',
    'has_auction_rating', 'auction_rating_filled'
]
corr = df[num_cols].corr(method='pearson')

plt.figure(figsize=(10, 7.5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0, linewidths=0.5
)
plt.title('Pearson Correlation Matrix (Numeric Features vs. Log Price)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# FIGURE 5: OLX Secondary Dataset — Top Car Features & Assembly/Document Impact
df_olx['price_lacs'] = df_olx['price_pkr'] / 100_000.0
tag_counts = (
    df_olx['car_features']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.barplot(x=tag_counts.values, y=tag_counts.index, hue=tag_counts.index, palette='mako', legend=False, ax=axes[0])
axes[0].set_title('Top 15 Listed Equipment Tags in OLX Dataset')
axes[0].set_xlabel('Number of Listings')
axes[0].set_ylabel('Feature Tag')

sns.boxplot(
    data=df_olx, x='assembly', y='price_lacs', hue='car_documents',
    showfliers=False, palette='Set1', ax=axes[1]
)
axes[1].set_title('OLX Price by Assembly (Local vs. Imported) & Document Status')
axes[1].set_xlabel('Assembly Type')
axes[1].set_ylabel('Price (PKR Lacs)')
axes[1].legend(title='Documents')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_olx_features_and_assembly.png', bbox_inches='tight')
plt.show()

In [ ]:
# Save quantitative EDA findings to reports/evidence/03_eda_insights.txt
summary_lines = [
    '===========================================================================',
    'AUTOPULSE STEP 3: QUANTITATIVE EDA SUMMARY',
    '===========================================================================',
    f'Total Clean Pakistan Used Cars Records: {len(df):,}',
    f'Raw Price Skewness: {df["price_pkr"].skew():.3f} | Log Price Skewness: {df["log_price"].skew():.3f}',
    f'Median Price: PKR {df["price_pkr"].median():,.0f} ({df["price_lacs"].median():.2f} Lacs)',
    f'IQR Price (25th - 75th): PKR {df["price_pkr"].quantile(0.25):,.0f} to {df["price_pkr"].quantile(0.75):,.0f}',
    '',
    '--- Top Correlations with log_price ---',
    corr['log_price'].sort_values(ascending=False).to_string(),
    '',
    '--- Median Price (PKR Lacs) by Fuel Type ---',
    df.groupby('fuel_type')['price_lacs'].agg(['count', 'median', 'mean']).sort_values('median', ascending=False).to_string(),
    '',
    '--- Median Price (PKR Lacs) by Transmission ---',
    df.groupby('transmission')['price_lacs'].agg(['count', 'median', 'mean']).to_string(),
]
report_text = '\n'.join(summary_lines)
(EVIDENCE_DIR / '03_eda_insights.txt').write_text(report_text, encoding='utf-8')
print(report_text)